In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("marquis03/high-resolution-viton-zalando-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/high-resolution-viton-zalando-dataset


In [ ]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.31.1
    Uninstalling protobuf-6.31.1:
      Successfully uninstalled protobuf-6.31.1


In [ ]:
# https://github.com/rlawjdghek/StableVITON/tree/master
import os
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
from tensorflow import keras
import json

In [ ]:
os.listdir(path)

['train_pairs.txt', 'test_pairs.txt', 'test', 'train']

In [ ]:
train = os.path.join(path, 'train')
test = os.path.join(path, 'test')

In [ ]:
train

'/kaggle/input/high-resolution-viton-zalando-dataset/train'

Memory management code


In [ ]:
# import tensorflow as tf
from tensorflow.keras import mixed_precision

# Set mixed precision policy
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

In [ ]:
# print(mixed_precision.global_policy())
# Should output: <Policy "mixed_float16">


In [ ]:
# class CrossAttention(keras.layers.Layer):
#   def __init__(self, units, num_heads = 8):
#     super(CrossAttention, self).__init__()
#     self.units = units
#     self.num_heads= num_heads
#     self.depth = units//num_heads

#     self.query = keras.layers.Dense(units=units)
#     self.Key = keras.layers.Dense(units=units)
#     self.value = keras.layers.Dense(units=units)
#     self.combine_heads = keras.layers.Dense(units=units)

#   def _split_head(self, inputs, batch_size):
#     x = tf.reshape(inputs, (batch_size, -1, self.num_heads, self.depth))
#     return tf.transpose(x, perm=[0,2,1,3])

#   def _combine_head(self, inputs, batch_size):
#     # print("combine Head", inputs.shape)
#     x = tf.transpose(inputs, perm=[0,2,1,3])
#     # print("combine Head transpose", inputs.shape)
#     return tf.reshape(x, (batch_size, -1, self.units))

#   def call(self, queries, keys, values):
#     batch_size = tf.shape(queries)[0]
#     height = tf.shape(queries)[1]
#     width = tf.shape(queries)[2]
#     qrs = self.query(queries)
#     kys = self.Key(keys)
#     vls = self.value(values)

#     qrs = self._split_head(qrs, batch_size)
#     kys = self._split_head(kys, batch_size)
#     vls = self._split_head(vls, batch_size)

#     attention_score = tf.matmul(qrs, kys, transpose_b=True)/tf.math.sqrt(tf.cast(self.num_heads, tf.float32))
#     attention_weights = tf.nn.softmax(attention_score, axis=-1)
#     output = tf.matmul(attention_weights, vls)
#     output = self._combine_head(output, batch_size)
#     output = tf.reshape(output, (batch_size, height, width, self.units))
#     return self.combine_heads(output)

In [ ]:
import tensorflow as tf
from tensorflow import keras

class CrossAttention(keras.layers.Layer):
    def __init__(self, units, num_heads=8):
        super(CrossAttention, self).__init__()
        self.units = units
        self.num_heads = num_heads
        self.depth = units // num_heads

        self.query = keras.layers.Dense(units=units)
        self.key = keras.layers.Dense(units=units)
        self.value = keras.layers.Dense(units=units)
        self.combine_heads = keras.layers.Dense(units=units)

    def _split_head(self, inputs, batch_size):
        x = tf.reshape(inputs, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])  # [B, heads, seq_len, depth]

    def _combine_head(self, inputs, batch_size):
        x = tf.transpose(inputs, perm=[0, 2, 1, 3])  # [B, seq_len, heads, depth]
        return tf.reshape(x, (batch_size, -1, self.units))

    def _forward(self, queries, keys, values):
        batch_size = tf.shape(queries)[0]
        height = tf.shape(queries)[1]
        width = tf.shape(queries)[2]

        q = self.query(queries)
        k = self.key(keys)
        v = self.value(values)

        q = self._split_head(q, batch_size)
        k = self._split_head(k, batch_size)
        v = self._split_head(v, batch_size)

        score = tf.matmul(q, k, transpose_b=True)
        score = score / tf.math.sqrt(tf.cast(self.depth, score.dtype))
        weights = tf.nn.softmax(score, axis=-1)
        attn = tf.matmul(weights, v)

        out = self._combine_head(attn, batch_size)
        out = tf.reshape(out, (batch_size, height, width, self.units))
        return self.combine_heads(out)

    def call(self, queries, keys, values):
        @tf.custom_gradient
        def run_with_checkpoint(q, k, v):
            def grad_fn(dy, *, variables=None):
                with tf.GradientTape() as tape:
                    tape.watch([q, k, v])
                    y = self._forward(q, k, v)
                grads = tape.gradient(dy, [q, k, v] + variables)
                return (grads[0], grads[1], grads[2]), grads[3:]

            y = self._forward(q, k, v)
            return y, grad_fn

        return run_with_checkpoint(queries, keys, values)

In [ ]:
class SelfAttention(keras.layers.Layer):
  def __init__(self, units):
    super(SelfAttention, self).__init__()
    self.attention = CrossAttention(units=units)

  def call(self, x):
    return self.attention(x, x, x)

In [ ]:
class DualPathConditionalUnet(keras.models.Model):
  def __init__(self, image_size = (64,64) , base_channels = 32):
    super(DualPathConditionalUnet, self).__init__()
    self.image_size = image_size
    self.base_channels = base_channels

    self.g_encoder = self._build_encoder("garment_encoder")
    self.p_encoder = self._build_encoder("person_encoder")
    self.bottleneck = self._build_bottleneck()
    self.decoder = self._build_decoder()

    self.time_mlp = keras.models.Sequential([
        keras.layers.Dense(units=self.base_channels*4, activation="swish"),
        keras.layers.Dense(units=self.base_channels*4)
    ])

    self.clip_proj = keras.layers.Dense(units=self.base_channels*4)

  def _build_encoder(self, name):
    inputs = keras.layers.Input(shape=(*self.image_size, 3 if name == "garment_encoder" else 27))
    x = inputs

    down_blocks = []
    channels = self.base_channels

    for i in range(4):
      x = keras.layers.Conv2D(filters=channels, kernel_size=3, padding='same')(x)
      x = keras.layers.GroupNormalization(groups=8)(x)
      x = keras.layers.ReLU()(x)

      x = SelfAttention(channels)(x)
      down_blocks.append(x)
      x = keras.layers.Conv2D(filters=channels, kernel_size=3, strides=2, padding='same')(x)
      x = keras.layers.GroupNormalization(groups=8)(x)
      x = keras.layers.ReLU()(x)
      print("Encoder Loop Channels/layers shapes",channels)
      channels *= 2


    return keras.models.Model(inputs, [x, down_blocks], name = name)

  def _build_bottleneck(self):
    channels = self.base_channels * 8
    height = self.image_size[0] // 16
    width = self.image_size[1] // 16

    # Inputs
    g_feat = keras.layers.Input(shape=(height, width, channels))
    p_feat = keras.layers.Input(shape=(height, width, channels))
    time_emb_input = keras.layers.Input(shape=(self.base_channels * 4,))  # Keep original

    # Process time embedding
    print("Bottle Net layers shape", channels)
    time_emb = keras.layers.Dense(units=channels)(time_emb_input)
    time_emb = keras.layers.Reshape((1, 1, channels))(time_emb)
    time_emb = keras.layers.Lambda(
        lambda x: tf.tile(x, [1, height, width, 1]),
        output_shape=(height, width, channels)
    )(time_emb)

    # Attention paths
    g_attn = CrossAttention(channels)(p_feat, g_feat, g_feat)
    p_attn = CrossAttention(channels)(g_feat, p_feat, p_feat)

    # Concatenate and process
    x = keras.layers.Concatenate()([g_attn, p_attn, time_emb])
    x = keras.layers.Conv2D(filters=channels, kernel_size=3, strides=1, padding='same')(x)
    x = keras.layers.GroupNormalization(groups=8)(x)
    x = keras.layers.ReLU()(x)

    # Return model with correct input tensors
    return keras.models.Model([g_feat, p_feat, time_emb_input], x)

  def _build_decoder(self):
    channels = self.base_channels*8
    print("Decoder startup channels",channels)
    inputs = keras.layers.Input(shape=(self.image_size[0]//16, self.image_size[1]//16, channels))

    x = inputs

    skip_inputs = []
    # skip_channels = [128, 256, 512, 1024]
    skip_channels = [64, 128, 256, 512]
    # skip_channels = [x.shape[-1] for x in g_skips]
    for _ in skip_channels:
      skip_inputs.append(keras.layers.Input(shape=(None, None, _ )))#channels * 2
      print("Decoder skiping channel", channels)
      channels //= 2 #channels *= 2

    for i in range(4):
      x = keras.layers.Conv2DTranspose(filters=channels, kernel_size=3, strides=2, padding='same')(x)
      print("layer_shape", x.shape)
      x = keras.layers.GroupNormalization(groups=8)(x)
      x = keras.layers.ReLU()(x)

      skip = skip_inputs[3-i]
      if skip.shape[1:3] != x.shape[1:3]:
        skip = keras.layers.Lambda(lambda tensors: tf.image.resize(tensors[0], tf.shape(tensors[1])[1:3]))([skip, x])

      x = keras.layers.Concatenate()([x, skip])

      x = keras.layers.Conv2D(filters = channels, kernel_size=3, padding='same')(x)
      x = keras.layers.GroupNormalization(groups=8)(x)
      x = keras.layers.ReLU()(x)

      x = SelfAttention(channels)(x)
      channels *= 2
    outputs = keras.layers.Conv2D(filters=3, kernel_size=3, padding='same', dtype='float32')(x)
    return keras.models.Model([inputs, *skip_inputs], outputs, name = "Decoder")


  def call(self, inputs):

    person_image = inputs["person_image"]
    cloth_image = inputs["cloth_image"]
    cloth_mask = inputs["cloth_mask"]
    parse_agnostic = inputs["parse_agnostic"]
    densepose = inputs["densepose"]
    pose_map = inputs["pose_map"]
    time = inputs['time']
    text_emb = inputs.get('text_emb', None)

    person_features = tf.concat([
        person_image*(1-parse_agnostic),
        parse_agnostic,
        densepose,
        pose_map
        ], axis=-1)

    time_emb = self.time_mlp(time)

    if text_emb is not None:
      text_proj = self.clip_proj(text_emb)
      time_emb = time_emb + text_proj
    # Encoder feature
    g_feat, g_skips = self.g_encoder(cloth_image*cloth_mask)
    p_feat, p_skips = self.p_encoder(person_features)


    bottleneck = self.bottleneck([g_feat, p_feat, time_emb])

    skips =  [tf.concat([g, p], axis=-1) for g, p in zip(g_skips, p_skips)]
    outputs = self.decoder([bottleneck, *skips])
    return outputs

In [ ]:
class GarmentTransferDiffusion(keras.models.Model):
  def __init__(self, image_size = (128,128), noise_steps = 1000):
    super(GarmentTransferDiffusion, self).__init__()
    self.image_size = image_size
    self.noise_steps = noise_steps

    self.beta = tf.linspace(1e-4, 0.02, self.noise_steps)
    self.alpha = 1.0 - self.beta
    self.alpha_bar = tf.math.cumprod(self.alpha, axis=0)

    self.unet = DualPathConditionalUnet(image_size)
    self.clip_model = None
    self.optimizer=keras.optimizers.Adam(learning_rate=args.learning_rate)

  def _get_time_embedding(self, time):
    half_dim = self.unet.base_channels * 4 //2
    emb = tf.math.log(10000.0)/(half_dim-1)
    emb = tf.exp(tf.cast(tf.range(half_dim), tf.float32) * -emb)
    emb = tf.cast(time, tf.float32)[:, None] * emb[None, :]
    emb = tf.concat([tf.sin(emb), tf.cos(emb)], axis=1)
    return emb
  def call(self, inputs):

    noise = tf.random.normal(tf.shape(inputs["person_image"]), dtype=inputs["person_image"].dtype)
    time = tf.random.uniform([tf.shape(inputs['person_image'])[0]], 0, self.noise_steps, dtype=tf.int32)

    alpha_bar = tf.gather(self.alpha_bar, time)
    alpha_bar = tf.cast(alpha_bar, dtype=inputs["person_image"].dtype)
    alpha_bar = tf.reshape(alpha_bar, [-1,1,1,1])
    # noisy_image = tf.sqrt(alpha_bar) * inputs["person_image"] + tf.sqrt(1 - alpha_bar) * noise
    noisy_image = tf.sqrt(alpha_bar) * inputs["person_image"] + tf.sqrt(1-alpha_bar) * noise

    inputs['time'] = self._get_time_embedding(time)
    inputs['person_image'] = noisy_image
    return self.unet(inputs)

  def train_step(self, inputs):
    with tf.GradientTape() as tape:
      predicted_noise = self(inputs)
      noise = inputs["noise"]

      loss = tf.reduce_mean(tf.square(predicted_noise - noise))

    grads = tape.gradient(loss, self.trainable_weights)
    self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
    return {"loss": loss}

  # def build_model(self):
  #   # Create sample inputs to build the model
  #   sample_input = {
  #       "person_image": Input(shape=(*self.image_size, 3)),
  #       "cloth_image": Input(shape=(*self.image_size, 3)),
  #       "cloth_mask": Input(shape=(*self.image_size, 1)),
  #       "parse_agnostic": Input(shape=(*self.image_size, 1)),
  #       "densepose": Input(shape=(*self.image_size, 3)),
  #       "pose_map": Input(shape=(*self.image_size, 18)),
  #       "time": Input(shape=(self.unet.base_channels*4,)),
  #       "text_emb": Input(shape=(512,)),  # Example CLIP embedding size
  #   }
  #   return Model(inputs=sample_input, outputs=self.call(sample_input))

In [ ]:
class GarmentSampler:
  def __init__(self, model, image_size = (128,128), noise_steps = 1000):
    self.model = model
    self.image_size = image_size
    self.noise_steps = noise_steps

    self.beta = model.beta
    self.alpha = model.alpha
    self.alpha_bar = model.alpha_bar

  def sample(self, inputs, steps = 250, guidance_scale = 3.0):
    x_t = tf.random.normal(shape=(tf.shape(inputs["person_image"])[0], *self.image_size, 3), dtype=tf.float32)
    step_size = self.noise_steps // steps
    timesteps = reversed(range(0, self.noise_steps, step_size))

    for t in timesteps:
      t_tensor = tf.fill([tf.shape(x_t)[0]], t)
      time_emb = self.model._get_time_embedding(t_tensor)

      model_inputs = {
          **inputs,
          'person_image': x_t,
          'time': time_emb
      }
      predicted_noise = self.model.unet(model_inputs, training=False)

      if guidance_scale > 1.0:
        uncond_inputs = {
            **inputs,
            'person_image' : x_t,
            'time' : self.model._get_time_embedding(tf.fill([tf.shape(x_t)[0]], self.noise_steps))
        }

        uncond_predicted_noise = self.model.unet(uncond_inputs, training=False)
        predicted_noise = uncond_predicted_noise + guidance_scale * (predicted_noise - uncond_predicted_noise)

    alpha_t = tf.gather(self.alpha, t_tensor)
    alpha_bar_t = tf.gather(self.alpha_bar, t_tensor)

    alpha_t = tf.reshape(alpha_t, [-1,1,1,1])
    alpha_bar_t = tf.reshape(alpha_bar_t, [-1,1,1,1])

    if t>0:
      noise = tf.random.normal(tf.shape(x_t))
    else:
      noise = tf.zeros_like(x_t)

    x_t = 1 / tf.sqrt(alpha_t) * (x_t - ((1-alpha_t) / tf.sqrt(1-alpha_bar_t)) * predicted_noise) + tf.sqrt(1-alpha_t) * noise
    result = x_t * inputs['cloth_mask'] + inputs['person_image'] * (1-inputs['cloth_mask'])
    return tf.clip_by_value(result, -1.0, 1.0)


In [ ]:
import argparse
import kagglehub
import os
import sys

class configure:
  def __init__(self):
    dataset_path = kagglehub.dataset_download("marquis03/high-resolution-viton-zalando-dataset")
    self.train = os.path.join(dataset_path, 'train')
    self.test = os.path.join(dataset_path, 'test')
    self.parser = argparse.ArgumentParser()
    self.parser.add_argument('--train_dir', type=str, default=self.train, help="Path to Train Data Directory")
    self.parser.add_argument('--image_size', type=int, nargs=2, default=(128,128), help="Image Size")
    self.parser.add_argument('--noise_steps', type=int, default=1000, help="Noise Steps")
    self.parser.add_argument('--batch_size', type=int, default=8, help="Batch Size")
    self.parser.add_argument('--epochs', type=int, default=1, help="Here is the number of training epochs")
    self.parser.add_argument('--learning_rate', type=float, default=1e-4, help="Model Learning Rate")
    self.parser.add_argument('--test_dir', type=str, default=self.test, help="test data directory")
    self.parser.add_argument('--base_image_size', type=int, nargs=2, default=(768,1024), help="Base Image Size")
    # self.args = self.parser.parse_args()
    if "ipykernel" in sys.modules:
      self.args = self.parser.parse_args([])
    else:
      self.args = self.parser.parse_args()
  def get_args(self):
    return self.args

In [ ]:
if __name__ == "__main__":
  config = configure()
  args = config.get_args()
  # train_diffusin_model(args)

In [ ]:
from sklearn.model_selection import train_test_split


class DataLoader:
  def __init__(self, data_dir, image_size=args.image_size, batch_size=args.batch_size, base_image_size = args.base_image_size):
    self.data_dir = data_dir
    self.image_size = image_size
    self.batch_size = batch_size
    self.base_image_size = base_image_size

  def _load_image(self, path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, self.image_size)
    return (tf.cast(image, dtype=tf.float32)/127.5)-1.0

  def _load_mask(self, path):
    mask = tf.io.read_file(path)
    mask = tf.image.decode_jpeg(mask, channels=1)
    mask = tf.image.resize(mask, self.image_size)
    return tf.cast(mask, dtype=tf.float32)/255.0

  def _parse_pose(self, json_path):
    with open(json_path.numpy().decode('utf-8')) as f:
      pose_data = json.load(f)
    keypoints = np.array(pose_data['people'][0]['pose_keypoints_2d']).reshape(-1, 3)[:, :2]
    pose_map = np.zeros((*self.image_size, 18), dtype=np.float32)
    for i, (x,y) in enumerate(keypoints[:18]):
      if x > 0 and y > 0:
        xx = int(x*self.image_size[0]/self.base_image_size[0])
        yy = int(y*self.image_size[1]/self.base_image_size[1])
        if 0<=xx<self.image_size[1] and 0<=yy<self.image_size[0]:
          pose_map[yy,xx,i] = 1.0
    return pose_map

  def _process_sample(self, sample_id):
    # sample_id = sample_id.numpy().decode("utf-8")
    base_path = lambda x, y: tf.strings.join([self.data_dir, "/", x, "/", sample_id, y])
    json_path = tf.strings.join([self.data_dir, "/openpose_json/", sample_id, "_keypoints.json"])
    print(type(json_path))

    person_image = self._load_image(base_path("image", ".jpg"))
    cloth_image = self._load_image(base_path("cloth", ".jpg"))

    cloth_mask = self._load_mask(base_path("cloth-mask", ".jpg"))
    parse_agnostic = self._load_image(base_path("image-parse-agnostic-v3.2", ".png"))
    densepose = self._load_image(base_path("image-densepose", ".jpg"))

    pose_map = tf.py_function(self._parse_pose, [json_path], tf.float32)
    pose_map.set_shape((*self.image_size, 18))

    noise = tf.random.normal(tf.shape(person_image))
    return {
        "person_image": person_image,
        "cloth_image": cloth_image,
        "cloth_mask": cloth_mask,
        "parse_agnostic": parse_agnostic,
        "densepose": densepose,
        "pose_map": pose_map,
        "noise": noise,
    }
  def create_dataset(self):
    sample_ids = [f.split('.')[0] for f in os.listdir(os.path.join(self.data_dir, 'image'))]
    train_ids, val_ids = train_test_split(sample_ids, test_size=0.2, random_state=42)
    train_ds = tf.data.Dataset.from_tensor_slices(train_ids)
    train_da = train_ds.shuffle(1000).map(self._process_sample, num_parallel_calls=tf.data.AUTOTUNE).batch(self.batch_size).prefetch(tf.data.AUTOTUNE)

    val_da = tf.data.Dataset.from_tensor_slices(val_ids).map(self._process_sample, num_parallel_calls=tf.data.AUTOTUNE).batch(self.batch_size).prefetch(tf.data.AUTOTUNE)
    return train_da, val_da


In [ ]:
dataset = DataLoader(data_dir=train)
train_data, val_data = dataset.create_dataset()

model = GarmentTransferDiffusion(image_size=args.image_size, noise_steps=args.noise_steps)
# Test loading
try:
    for batch in train_data.take(2):
        for key, value in batch.items():
            print(f"{key}: shape={value.shape}, dtype={value.dtype}")
        # losess = model.train_step(batch)
except Exception as e:
    print(f"Error loading data: {e}")

<class 'tensorflow.python.framework.ops.SymbolicTensor'>
<class 'tensorflow.python.framework.ops.SymbolicTensor'>
Encoder Loop Channels/layers shapes 32
Encoder Loop Channels/layers shapes 64
Encoder Loop Channels/layers shapes 128
Encoder Loop Channels/layers shapes 256
Encoder Loop Channels/layers shapes 32
Encoder Loop Channels/layers shapes 64
Encoder Loop Channels/layers shapes 128
Encoder Loop Channels/layers shapes 256
Bottle Net layers shape 256
Decoder startup channels 256
Decoder skiping channel 256
Decoder skiping channel 128
Decoder skiping channel 64
Decoder skiping channel 32
layer_shape (None, 16, 16, 16)
layer_shape (None, 32, 32, 32)
layer_shape (None, 64, 64, 64)
layer_shape (None, 128, 128, 128)
person_image: shape=(8, 128, 128, 3), dtype=<dtype: 'float32'>
cloth_image: shape=(8, 128, 128, 3), dtype=<dtype: 'float32'>
cloth_mask: shape=(8, 128, 128, 1), dtype=<dtype: 'float32'>
parse_agnostic: shape=(8, 128, 128, 3), dtype=<dtype: 'float32'>
densepose: shape=(8, 128,

In [ ]:
losess = model.train_step(batch)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'dual_path_conditional_unet', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/optimizers/base_optimizer.py:774: UserWarning: Gradients do not exist for variables ['conv2d/kernel', 'conv2d/bias', 'group_normalization/gamma', 'group_normalization/beta', 'self_attention/cross_attention/dense/kernel', 'self_attention/cross_attention/dense/bias', 'self_attention/cross_attention/dense_1/kernel', 'self_attention/cross_attention/dense_1/bias', 'self_attention/cross_attention/dense_2/kernel', 'self_attention/cross_attention/dense_2/bias', 'self_attention/cross_attention/dense_3/kernel', 'self_

In [ ]:
import gc
tf.keras.backend.clear_session()
gc.collect()

0

In [ ]:
losess

{'loss': <tf.Tensor: shape=(), dtype=float32, numpy=2.024980306625366>}

In [ ]:
tf.config.run_functions_eagerly(True)

In [ ]:
import tensorflow as tf
print(tf.executing_eagerly())  # Should print: True


True


In [ ]:
 OUT = []

 @tf.function #(jit_compile=True)
 def train_diffusin_model():
  config = {
      'image_size': (128,128),
      'noise_steps': 1000,
      'batch_size': 16,
      'epochs': 1,
      'data_dir': train,
      'learning_rate': 1e-4,
      'noise_steps': 1000
  }
  data_loader = DataLoader(args.train_dir, image_size=args.image_size, batch_size=args.batch_size)
  train_dataset, val_dataset = data_loader.create_dataset()
  model = GarmentTransferDiffusion(image_size=args.image_size, noise_steps=args.noise_steps)
  model.compile(optimizer=keras.optimizers.Adam(learning_rate=args.learning_rate))

  sampler = GarmentSampler(model, image_size=args.image_size, noise_steps=args.noise_steps)

  for epochs in range(config['epochs']):
    print(f"\nEpoch {epochs+1} / {config['epochs']} ")

    for step, batch in enumerate(train_dataset):
      losses = model.train_step(batch)

      if step%100==0:
        tf.print("Step", step, ": losses =", losses['loss'])

    if epochs % 5 == 0:
      for val_batch in val_dataset.take(1):
        sample = sampler.sample(val_batch, steps = 250)

        sample_image = tf.clip_by_value((sample[0] + 1) * 127.5, 0, 255)
        sample_image = tf.cast(sample_image, tf.uint8).numpy()
        OUT.append(sample_image)

        # cv2.imwrite(f"sample_epoch_{epochs}.png", cv2.cvtColor(sample_image, cv2.COLOR_RGB2BGR))

    model.save_weights("dual_path_diffusion_model.weights.h5")
    model.save_weights("dual_path_diffusion_model.weights.keras")


In [ ]:
train_diffusin_model()

/usr/local/lib/python3.11/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


<class 'tensorflow.python.framework.ops.SymbolicTensor'>
<class 'tensorflow.python.framework.ops.SymbolicTensor'>
Encoder Loop Channels/layers shapes 32
Encoder Loop Channels/layers shapes 64
Encoder Loop Channels/layers shapes 128
Encoder Loop Channels/layers shapes 256
Encoder Loop Channels/layers shapes 32
Encoder Loop Channels/layers shapes 64
Encoder Loop Channels/layers shapes 128
Encoder Loop Channels/layers shapes 256
Bottle Net layers shape 256
Decoder startup channels 256
Decoder skiping channel 256
Decoder skiping channel 128
Decoder skiping channel 64
Decoder skiping channel 32
layer_shape (None, 16, 16, 16)
layer_shape (None, 32, 32, 32)
layer_shape (None, 64, 64, 64)
layer_shape (None, 128, 128, 128)

Epoch 1 / 1 


/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'dual_path_conditional_unet_4', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


In [ ]:
import tensorflow.keras.backend as K
K.eval(OUT[0])

In [ ]:
# opt = mixed_precision.LossScaleOptimizer(opt)


In [ ]:
# @tf.function
# def train_diffusion_model():
#     config = {
#         'image_size': (128, 128),
#         'noise_steps': 1000,
#         'batch_size': 4,  # Use bigger batch size
#         'accumulation_steps': 4,  # Number of micro-batches to accumulate
#         'epochs': 100,
#         'data_dir': train,
#         'learning_rate': 1e-4,
#     }
#     data_loader = DataLoader(config['data_dir'], image_size=config['image_size'], batch_size=config['batch_size'])
#     train_dataset, val_dataset = data_loader.create_dataset()

#     model = GarmentTransferDiffusion(image_size=config['image_size'], noise_steps=config['noise_steps'])
#     optimizer = keras.optimizers.Adam(learning_rate=config['learning_rate'])
#     optimizer = mixed_precision.LossScaleOptimizer(optimizer)

#     sampler = GarmentSampler(model, image_size=config['image_size'], noise_steps=config['noise_steps'])

#     for epoch in range(config['epochs']):
#         print(f"\nEpoch {epoch + 1} / {config['epochs']}")

#         for step, batch in enumerate(train_dataset):
#             # Split batch into micro-batches
#             micro_batches = tf.split(batch, num_or_size_splits=config['accumulation_steps'])

#             accumulated_grads = [tf.zeros_like(var) for var in model.trainable_variables]
#             loss_value = 0.0

#             for micro_batch in micro_batches:
#                 with tf.GradientTape() as tape:
#                     loss = model.train_step(micro_batch)
#                 grads = tape.gradient(loss, model.trainable_variables)
#                 accumulated_grads = [acc_grad + (grad if grad is not None else 0)
#                                      for acc_grad, grad in zip(accumulated_grads, grads)]
#                 loss_value += loss / config['accumulation_steps']

#             # Average gradients and apply optimizer step
#             averaged_grads = [grad / config['accumulation_steps'] for grad in accumulated_grads]
#             optimizer.apply_gradients(zip(averaged_grads, model.trainable_variables))

#             if step % 100 == 0:
#                 print(f"Step {step}: loss = {loss_value:.4f}")

#         if epoch % 5 == 0:
#             for val_batch in val_dataset.take(1):
#                 sample = sampler.sample(val_batch, steps=250)
#                 sample_image = ((sample[0].numpy() + 1) * 127.5).astype(np.uint8)
#                 cv2.imwrite(f"sample_epoch_{epoch}.png", cv2.cvtColor(sample_image, cv2.COLOR_RGB2BGR))

#         model.save_weights("dual_path_diffusion_model.weights.h5")
#         model.save_weights("dual_path_diffusion_model.weights.keras")


In [ ]:
import kagglehub

# Download latest version
# path = kagglehub.dataset_download("kaggle/meta-kaggle-code")

# print("Path to dataset files:", path)


Path to dataset files: /root/.cache/kagglehub/datasets/marquis03/high-resolution-viton-zalando-dataset/versions/1


In [ ]:
# config = {
#     'image_size': (128,128),
#     'noise_steps': 1000,
#     'batch_size': 1,
#     'epochs': 100,
#     'data_dir': train,
#     'learning_rate': 1e-4,
#     'noise_steps': 1000
# }
# data_loader = DataLoader(args.train_dir, image_size=args.image_size, batch_size=args.batch_size)
# train_dataset, val_dataset = data_loader.create_dataset()
# model = GarmentTransferDiffusion(image_size=args.image_size, noise_steps=args.noise_steps)
# model.compile(optimizer=keras.optimizers.Adam(learning_rate=args.learning_rate))

# sampler = GarmentSampler(model, image_size=args.image_size, noise_steps=args.noise_steps)


# if epochs % 5 == 0:
#   for val_batch in val_dataset.take(1):
#     sample = sampler.sample(val_batch, steps = 10)

#     sample_image = ((sample[0].numpy() + 1) * 127.5).astype(np.uint8)
#     cv2.imwrite(f"sample_epoch_{epochs}.png", cv2.cvtColor(sample_image, cv2.COLOR_RGB2BGR))

NameError: name 'train' is not defined

In [ ]:
args.base_image_size

(768, 1024)

In [ ]:
sampler = GarmentSampler(model, image_size=args.image_size, noise_steps=args.noise_steps)

In [ ]:
for val_batch in val_data.take(1):
  pass

In [ ]:
sample = sampler.sample(val_batch, steps = 250)